In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        (os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import os, cv2, numpy as np, pandas as pd
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import DenseNet121, ResNet50

2026-04-13 05:19:19.723986: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776057559.924377     160 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776057559.983611     160 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776057560.457686     160 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776057560.457729     160 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776057560.457732     160 computation_placer.cc:177] computation placer alr

In [3]:
df = pd.read_csv("/kaggle/input/datasets/amimulahasanrofik/abu-csv/data_info.csv")

orig_dir = "/kaggle/input/datasets/amimulahasanrofik/abu-sayed/Fundus_CIMT_2903 Dataset"
vessel_dir = "/kaggle/input/datasets/amimulahasanrofik/vesselenhancedmaps"

df['orig_path'] = df['image_name'].apply(lambda x: os.path.join(orig_dir, x))
df['vessel_path'] = df['image_name'].apply(lambda x: os.path.join(vessel_dir, x))

In [4]:
IMG_SIZE = 224

def load_images(paths):
    imgs = []
    for p in tqdm(paths):
        img = cv2.imread(p)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        img = img / 255.0
        imgs.append(img)
    return np.array(imgs)

X_orig = load_images(df['orig_path'])
X_vessel = load_images(df['vessel_path'])

100%|██████████| 5806/5806 [00:56<00:00, 102.53it/s]


In [5]:
tab_cols = ['thickness', 'age', 'gender', 'group']

X_tab = df[tab_cols].values

scaler = StandardScaler()
X_tab = scaler.fit_transform(X_tab)

In [6]:
le = LabelEncoder()
y = le.fit_transform(df['label'])

num_classes = len(np.unique(y))

In [7]:
Xo_train, Xo_test, Xv_train, Xv_test, Xt_train, Xt_test, y_train, y_test = train_test_split(
    X_orig, X_vessel, X_tab, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [8]:
input_orig = layers.Input(shape=(224,224,3))
input_vessel = layers.Input(shape=(224,224,3))
input_tab = layers.Input(shape=(X_tab.shape[1],))

In [10]:
from tensorflow.keras.applications import DenseNet121, ResNet50

# Original → DenseNet
base1 = DenseNet121(
    weights='imagenet',
    include_top=False,
    input_tensor=input_orig
)

# Vessel → ResNet (🔥 better)
base2 = ResNet50(
    weights='imagenet',
    include_top=False,
    input_tensor=input_vessel
)

I0000 00:00:1776057752.259476     160 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [20]:
from tensorflow.keras import layers, models

# 🔷 Freeze layers
for layer in base1.layers:
    layer.trainable = False

for layer in base2.layers:
    layer.trainable = False


# 🔷 Original Image Branch (DenseNet)
x1 = base1.output
x1 = layers.GlobalAveragePooling2D()(x1)
x1 = layers.BatchNormalization()(x1)


# 🔷 Vessel Image Branch (ResNet)
x2 = base2.output
x2 = layers.GlobalAveragePooling2D()(x2)
x2 = layers.BatchNormalization()(x2)

In [21]:
input_tab = layers.Input(shape=(X_tab.shape[1],))

x3 = layers.Dense(64, activation='relu')(input_tab)
x3 = layers.BatchNormalization()(x3)
x3 = layers.Dropout(0.3)(x3)

x3 = layers.Dense(32, activation='relu')(x3)

In [22]:
# Combine all features
combined = layers.Concatenate()([x1, x2, x3])

# Attention mechanism
attention = layers.Dense(combined.shape[-1], activation='sigmoid')(combined)
combined = layers.Multiply()([combined, attention])

In [23]:
x = layers.Dense(512, activation='relu')(combined)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.6)(x)

x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.5)(x)

x = layers.Dense(128, activation='relu')(x)

output = layers.Dense(num_classes, activation='softmax')(x)

In [24]:
model = models.Model(
    inputs=[input_orig, input_vessel, input_tab],
    outputs=output
)

ValueError: The name "conv1_conv" is used 2 times in the model. All operation names should be unique.

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(patience=3)
]

In [16]:
x3 = layers.Dense(64, activation='relu')(input_tab)
x3 = layers.BatchNormalization()(x3)
x3 = layers.Dropout(0.3)(x3)

x3 = layers.Dense(32, activation='relu')(x3)

In [17]:
combined = layers.Concatenate()([x1, x2, x3])

# Attention weights
attention = layers.Dense(combined.shape[-1], activation='sigmoid')(combined)
combined = layers.Multiply()([combined, attention])

In [18]:
x = layers.Dense(512, activation='relu')(combined)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.6)(x)

x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.5)(x)

x = layers.Dense(128, activation='relu')(x)

output = layers.Dense(num_classes, activation='softmax')(x)

In [19]:
model = models.Model(
    inputs=[input_orig, input_vessel, input_tab],
    outputs=output
)

ValueError: The name "conv1_conv" is used 2 times in the model. All operation names should be unique.

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    [Xo_train, Xv_train, Xt_train],
    y_train,
    validation_data=([Xo_test, Xv_test, Xt_test], y_test),
    epochs=20,
    batch_size=8
)

In [ ]:
for layer in base1.layers[-20:]:
    layer.trainable = True

for layer in base2.layers[-20:]:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(
    [Xo_train, Xv_train, Xt_train],
    y_train,
    validation_data=([Xo_test, Xv_test, Xt_test], y_test),
    epochs=10,
    batch_size=8
)

In [ ]:
from sklearn.metrics import classification_report

y_pred = model.predict([Xo_test, Xv_test, Xt_test])
y_pred = np.argmax(y_pred, axis=1)

print(classification_report(y_test, y_pred))